In [0]:
%sql
-- ============================================================
-- TABELA: SIGA - EMPREENDIMENTOS DE GERAÇÃO
-- ORIGEM: ANEEL - Agência Nacional de Energia Elétrica
-- DATASET: siga-empreendimentos-geracao
-- FORMATO: Delta Lake
-- FREQUÊNCIA DE ATUALIZAÇÃO: Mensal
-- DICIONÁRIO: Versão 1.2 - 30/03/2023
-- ============================================================

CREATE TABLE IF NOT EXISTS mba.raw.usinas
(
    -- ========================================================
    -- IDENTIFICAÇÃO E CONTROLE DO DATASET
    -- ========================================================

    DatGeracaoConjuntoDados TIMESTAMP
        COMMENT 'Data e hora do processamento de carga automática no momento da geração para publicação do conjunto de dados abertos.',

    -- ========================================================
    -- IDENTIFICAÇÃO DO EMPREENDIMENTO
    -- ========================================================

    NomEmpreendimento STRING
        COMMENT 'Nome do empreendimento de geração.',
    IdeNucleoCEG STRING
        COMMENT 'Núcleo do Código Único de Empreendimentos de Geração.',
    CodCEG STRING
        COMMENT 'Código Único de Empreendimentos de Geração no formato GGG.FF.UF.999999-D.VV. GGG representa o tipo de geração, FF a fonte, UF a UF principal, 999999 o núcleo do CEG, D o dígito verificador e VV a versão.',
    SigUFPrincipal STRING
        COMMENT 'UF principal do empreendimento.',
    -- ========================================================
    -- CARACTERÍSTICAS DA GERAÇÃO
    -- ========================================================

    SigTipoGeracao STRING
        COMMENT 'Abreviação do tipo de geração. Valores definidos pela ANEEL: UTN - Usina Termonuclear; UTE - Usina Termelétrica; UHE - Usina Hidrelétrica; UFV - Central Geradora Solar Fotovoltaica; PCH - Pequena Central Hidrelétrica; EOL - Central Geradora Eólica; CGU - Central Geradora Undi-elétrica; CGH - Central Geradora Hidrelétrica.',
    DscFaseUsina STRING
        COMMENT 'Fase atual do empreendimento, desde etapas anteriores à outorga até a revogação.',
    DscOrigemCombustivel STRING
        COMMENT 'Origem da fonte do empreendimento.',
    DscFonteCombustivel STRING
        COMMENT 'Tipo da fonte do empreendimento.',
    DscTipoOutorga STRING
        COMMENT 'Tipo de atuação do empreendimento: Registro, Autorização ou Concessão.',
    NomFonteCombustivel STRING
        COMMENT 'Nome da fonte ou combustível do empreendimento.',

    -- ========================================================
    -- OPERAÇÃO E POTÊNCIA
    -- ========================================================

    DatEntradaOperacao DATE
        COMMENT 'Data de entrada em operação da primeira unidade geradora do empreendimento.',
    MdaPotenciaOutorgadaKw DECIMAL(16,2)
        COMMENT 'Potência total outorgada ou registrada do empreendimento, em kW.',
    MdaPotenciaFiscalizadaKw DECIMAL(16,2)
        COMMENT 'Potência do empreendimento que está em operação. Caso a usina não esteja mais em operação, corresponde à última potência considerada em operação, em kW.',
    MdaGarantiaFisicaKw DECIMAL(20,1)
        COMMENT 'Garantia Física homologada pelo MME - Ministério de Minas e Energia, em kW.',

    -- ========================================================
    -- GERAÇÃO QUALIFICADA E GEOLOCALIZAÇÃO
    -- ========================================================

    IdcGeracaoQualificada STRING
        COMMENT 'Identifica se o empreendimento foi enquadrado como Geração Qualificada conforme critérios da Resolução Normativa nº 235, de 14 de novembro de 2006.',
    NumCoordNEmpreendimento STRING
        COMMENT 'Latitude aproximada, em grau decimal, do ponto centróide de localização do empreendimento.',
    NumCoordEEmpreendimento STRING
        COMMENT 'Longitude aproximada, em grau decimal, do ponto centróide de localização do empreendimento.',

    -- ========================================================
    -- VIGÊNCIA DA OUTORGA
    -- ========================================================

    DatInicioVigencia DATE
        COMMENT 'Data de início da vigência da outorga, caso houver.',
    DatFimVigencia DATE
        COMMENT 'Data de fim da vigência da outorga, caso houver.',

    -- ========================================================
    -- PROPRIEDADE E LOCALIZAÇÃO
    -- ========================================================

    DscPropriRegimePariticipacao STRING
        COMMENT 'Descritivo do percentual de participação por agente na propriedade do empreendimento e o regime de exploração do agente em relação a este empreendimento.',
    DscSubBacia STRING
        COMMENT 'Código e descrição da sub-bacia do rio onde está o empreendimento. Preenchido apenas para usinas com fontes hídricas.',
    DscMuninicpios STRING
        COMMENT 'Descrição dos municípios e estados onde está localizado o empreendimento.',

    -- ========================================================
    -- METADADOS TÉCNICOS DA INGESTÃO
    -- ========================================================

    NmArquivoCarga STRING
        COMMENT 'Nome do arquivo de origem utilizado na carga.',
    DatCarga TIMESTAMP
        COMMENT 'Data e hora em que o registro foi carregado na tabela Delta.'
)

USING DELTA

COMMENT 'Empreendimentos de geração de energia elétrica do parque gerador nacional, contendo usinas nas diversas fases, desde etapas anteriores às outorgas até a revogação. Fonte: ANEEL - SIGA. Dataset: siga-empreendimentos-geracao. Atualização mensal. Dicionário de Metadados versão 1.2 de 30/03/2023.';

In [0]:
# ============================================================
# CARGA - SIGA EMPREENDIMENTOS DE GERAÇÃO
# ANEEL
# ============================================================

from pyspark.sql import functions as F

# ============================================================
# 1. CAMINHO DO ARQUIVO DE ORIGEM
# ============================================================

caminho_arquivo = (
    "/Volumes/mba/stage/dados_bruto/usinas/"
    "siga-empreendimentos-geracao.csv"
)

# ============================================================
# 2. LEITURA DO CSV
# ============================================================

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")       # CORREÇÃO
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("inferSchema", "false")
    .load(caminho_arquivo)
)

# ============================================================
# 3. PADRONIZAÇÃO DOS TIPOS
# ============================================================

df_tratado = (
    df

    # --------------------------------------------------------
    # DATAS
    # --------------------------------------------------------
    # Data de geração do conjunto de dados
    .withColumn(
        "DatGeracaoConjuntoDados",
        F.to_timestamp(
            F.col("DatGeracaoConjuntoDados"),
            "yyyy-MM-dd"
        )
    )

    # Data de entrada em operação
    .withColumn(
        "DatEntradaOperacao",
        F.to_date(
            F.col("DatEntradaOperacao"),
            "yyyy-MM-dd"
        )
    )

    # Início da vigência
    .withColumn(
        "DatInicioVigencia",
        F.to_date(
            F.col("DatInicioVigencia"),
            "yyyy-MM-dd"
        )
    )

    # Fim da vigência
    .withColumn(
        "DatFimVigencia",
        F.to_date(
            F.col("DatFimVigencia"),
            "yyyy-MM-dd"
        )
    )

    # --------------------------------------------------------
    # DECIMAIS
    # --------------------------------------------------------
    .withColumn(
        "MdaPotenciaOutorgadaKw",
        F.regexp_replace(
            F.col("MdaPotenciaOutorgadaKw"),
            ",",
            "."
        ).cast("decimal(16,2)")
    )

    .withColumn(
        "MdaPotenciaFiscalizadaKw",
        F.regexp_replace(
            F.col("MdaPotenciaFiscalizadaKw"),
            ",",
            "."
        ).cast("decimal(16,2)")
    )

    .withColumn(
        "MdaGarantiaFisicaKw",
        F.regexp_replace(
            F.col("MdaGarantiaFisicaKw"),
            ",",
            "."
        ).cast("decimal(20,1)")
    )
)

# ============================================================
# 4. SELEÇÃO E ORDEM DAS COLUNAS
# ============================================================

df_final = (
    df_tratado
    .withColumn(
        "NmArquivoCarga",
        F.lit("siga-empreendimentos-geracao.csv")
    )
    .withColumn(
        "DatCarga",
        F.current_timestamp()
    )
    .select(
        "DatGeracaoConjuntoDados",
        "NomEmpreendimento",
        "IdeNucleoCEG",
        "CodCEG",
        "SigUFPrincipal",
        "SigTipoGeracao",
        "DscFaseUsina",
        "DscOrigemCombustivel",
        "DscFonteCombustivel",
        "DscTipoOutorga",
        "NomFonteCombustivel",
        "DatEntradaOperacao",
        "MdaPotenciaOutorgadaKw",
        "MdaPotenciaFiscalizadaKw",
        "MdaGarantiaFisicaKw",
        "IdcGeracaoQualificada",
        "NumCoordNEmpreendimento",
        "NumCoordEEmpreendimento",
        "DatInicioVigencia",
        "DatFimVigencia",
        "DscPropriRegimePariticipacao",
        "DscSubBacia",
        "DscMuninicpios",
        "NmArquivoCarga",
        "DatCarga"
    )
)

# ============================================================
# 5. GRAVAÇÃO NA DELTA TABLE
# ============================================================
(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mba.raw.usinas")
)


In [0]:
dbutils.notebook.exit("OK")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM mba.raw.usinas
        LIMIT 20
    """)
)

# Quantidade de registros
spark.sql("""
    SELECT COUNT(*) AS quantidade_registros
    FROM mba.raw.usinas
""").show()

In [0]:
caminho_arquivo = (
    "/Volumes/mba/stage/dados_bruto/usinas/"
    "siga-empreendimentos-geracao.csv"
)

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")       # CORREÇÃO
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .option("inferSchema", "false")
    .load(caminho_arquivo)
).printSchema()